# GTHDM investigation

Scratch notebook to inspect the pymarty GTHDM port: particles, Lagrangian terms, and LaTeX output.

Setup: run the kernel with the pymarty build on `PYTHONPATH`, for example
`PYTHONPATH=/tmp/pymarty-build/python`, and a kernel that has `ipykernel` installed.
Building `GTHDM()` takes a few minutes; rerun the construction cell only after code changes.

The Lagrangian is grouped by `kind`: `"kinetic"`, `"mass"` and `"interaction"`.

In [ ]:
from IPython.display import Math, display

from pymarty.modeling.models import GTHDM
from pymarty.modeling.models._symbols import constant as shared_constant

model = GTHDM()
print(f"particles: {len(model.particles)}")

In [ ]:
print(sorted(p.name for p in model.particles))

In [ ]:
terms = {
    kind: model.lagrangian_terms(kind) for kind in ("kinetic", "mass", "interaction")
}
for kind, items in terms.items():
    print(f"{kind}: {len(items)} terms")

In [ ]:
# Show one term as LaTeX. Change the kind and index to inspect others.
kind, index = "mass", 5
display(Math(terms[kind][index].to_latex()))

In [ ]:
def top_level_terms(expr):
    """Split an expression string on ' + ' outside any brackets."""
    parts, depth, current, i = [], 0, "", 0
    text = str(expr)
    while i < len(text):
        ch = text[i]
        if ch in "([":
            depth += 1
        elif ch in ")]":
            depth -= 1
        if depth == 0 and text.startswith(" + ", i):
            parts.append(current)
            current, i = "", i + 3
            continue
        current += ch
        i += 1
    parts.append(current)
    return parts


def textual_cancellations(kind):
    """Yield (term index, component) for components whose negative also appears."""
    for idx, term in enumerate(terms[kind]):
        comps = top_level_terms(term)
        for comp in comps:
            neg = comp.startswith("-")
            key = comp[1:] if neg else comp
            if neg and key in comps and key != comp:
                yield idx, key


found = list(textual_cancellations("mass")) + list(textual_cancellations("interaction"))
print(f"textually cancelling pairs: {len(found)}")
for idx, key in found[:5]:
    print(idx, key[:150])

In [ ]:
# Inspect the summands of one term and check whether two of them cancel symbolically.
# The index 5 is an arbitrary example; set it to a term you want to examine.
expr = terms["mass"][5]
for arg in expr.args:
    print(str(arg)[:200])

pair_sum = expr.args[1] + expr.args[2]
print("pair sum expanded:", pair_sum.expand())

In [ ]:
# Compare the shared symbol used by the builders with a fresh pymarty constant.
from pymarty.symbolic import constant as fresh_constant

a = shared_constant("beta;\\beta")
b = shared_constant("beta;\\beta")
c = fresh_constant("beta;\\beta")
print("shared minus shared:", (a - b).expand())
print("shared minus fresh: ", (a - c).expand())

In [ ]:
# Free scratch cell.
